# Pelatihan Model Klasifikasi Tingkat Penumpukan Sampah (RAIDAS)

Notebook ini melatih model **CNN (transfer learning MobileNetV2)** untuk mengklasifikasikan kondisi saluran drainase ke dalam 3 kelas: **rendah**, **sedang**, dan **tinggi**.

**Alur:** memuat dataset → melihat isi data → augmentasi → membangun model → training → evaluasi → menyimpan model.

## 1. Import Library

Memuat TensorFlow/Keras dan MobileNetV2 sebagai model dasar (pretrained).

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import os

print("Versi TensorFlow:", tf.__version__)

## 2. Memuat Dataset

Dataset dibaca dari folder `dataset/train`. Nama subfolder otomatis menjadi nama kelas (`rendah`, `sedang`, `tinggi`), diurutkan abjad.

- Ukuran gambar: 224 x 224 piksel (tahap **Resize**)
- Batch size: 32
- Label: *categorical* (one-hot)

> Nama folder di `train` dan `val` harus sama persis.

In [ ]:
# Tentukan jalur folder dataset
TRAIN_DIR = '../dataset/train'
VAL_DIR = '../dataset/val'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Memuat data training secara otomatis dari folder
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Melihat nama kelas yang terdeteksi otomatis dari nama subfolder
class_names = train_dataset.class_names
print("Kelas terdeteksi:", class_names)

In [ ]:
# Memuat data validasi (tidak diacak urutannya, supaya hasil evaluasi konsisten)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

print("Kelas val:", val_dataset.class_names)
print("Sama dengan train:", val_dataset.class_names == class_names)

# Jumlah foto val per kelas
for nama in val_dataset.class_names:
    folder = os.path.join(VAL_DIR, nama)
    print(nama, ":", len(os.listdir(folder)))

## 3. Melihat Isi Data

Memeriksa jumlah foto per kelas dan menampilkan contoh gambar hasil resize untuk memastikan label sesuai isi gambar dan melihat efek perubahan proporsi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Hitung jumlah foto per kelas
for nama in class_names:
    folder = os.path.join(TRAIN_DIR, nama)
    print(nama, ":", len(os.listdir(folder)))

In [ ]:
# 2. Tampilkan 9 contoh gambar dari satu batch
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[np.argmax(labels[i])])
        plt.axis("off")
plt.show()

## 4. Augmentasi Data

Augmentasi membuat variasi gambar secara acak saat training agar model tidak menghafal foto dan lebih tahan terhadap perubahan cahaya dan sudut.

| Teknik | Fungsi |
|---|---|
| RandomFlip (horizontal) | Membalik gambar kiri-kanan |
| RandomRotation (0.05) | Memutar sedikit |
| RandomZoom (0.1) | Memperbesar/memperkecil |
| RandomBrightness dan RandomContrast (0.15) | Meniru variasi pencahayaan |

Augmentasi hanya aktif saat training, tidak saat prediksi.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name="augmentasi")

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    gambar = images[0]
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        hasil = data_augmentation(tf.expand_dims(gambar, 0), training=True)
        plt.imshow(hasil[0].numpy().astype("uint8"))
        plt.axis("off")
plt.show()

## 5. Membangun Model

Menggunakan **transfer learning MobileNetV2** (bobot ImageNet, dibekukan), dengan lapisan tambahan untuk 3 kelas.

**Susunan:** Input → Augmentasi → Normalisasi (-1 sampai 1) → MobileNetV2 → GlobalAveragePooling → Dropout (0.3) → Dense (128, ReLU) → Dense (3, softmax)

Normalisasi piksel dilakukan **di dalam model** sehingga server tidak perlu mengulangnya, dan perlakuan saat training dan saat prediksi pasti sama.

In [ ]:
NUM_CLASSES = len(class_names)

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,),
                         include_top=False,
                         weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 6. Training Model

> **Catatan:** kode training masih di-comment sampai dataset lengkap terkumpul (kelas sedang dan tinggi masih sedikit, dan folder `val` belum terisi).

Rencana yang akan ditambahkan:
- `validation_data` dari folder `dataset/val`
- `class_weight` untuk menyiasati ketimpangan jumlah foto antar kelas
- `EarlyStopping` agar training berhenti otomatis jika hasil validasi tidak membaik

In [ ]:
# EPOCHS = 10
# history = model.fit(
#     train_dataset,
#     epochs=EPOCHS
# )
print("Kerangka model sudah siap! Kode training di-comment dulu sampai dataset lengkap terkumpul.")